# Active Learning: Évaluation sur Domaine B et Sélection Intelligente de Données

## Pipeline:
1. **Charger** dataset et testSet du domaine B
2. **Évaluer** le checkpoint (DA) sur DB sans fine-tuning (baseline)
3. **Implémenter** les stratégies d'Active Learning (incertitude, diversité, MC dropout)
4. **Sélectionner** les K meilleures données à annoter
5. **Ré-entraîner** avec les données annotées sélectionnées
6. **Mesurer** l'amélioration

## Section 1: Charger le dataset et testSet du domaine B

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import cv2
from pathlib import Path

# Chemins
active_learning_dir = r'C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning'

# Chemins du domaine B
dataset_dir = os.path.join(active_learning_dir, 'Dataset')
testset_dir = os.path.join(active_learning_dir, 'testSet1')

# Charger dataset et testSet du domaine B (2 parties)
dataset_csv_path = os.path.join(dataset_dir, 'coords_SC1.csv')
dataset_csv_path_yas = os.path.join(dataset_dir, 'Coords_Yas1.csv')
testset_csv_path = os.path.join(testset_dir, 'coords_SC-test.csv')

print(f" Chemins:")
print(f"   Dataset CSV (SC1): {dataset_csv_path}")
print(f"   Dataset CSV (YAS1): {dataset_csv_path_yas}")
print(f"   TestSet CSV: {testset_csv_path}")

try:
    dataset_sc1 = pd.read_csv(dataset_csv_path)
    dataset_yas = pd.read_csv(dataset_csv_path_yas)
    testset_db = pd.read_csv(testset_csv_path)

    # Tag sources and image prefixes
    dataset_sc1['source'] = 'sc1'
    dataset_sc1['img_prefix'] = 'SC1'
    dataset_yas['source'] = 'yas1'
    dataset_yas['img_prefix'] = 'SCtes_t'

    # Merge both parts into one training dataset
    dataset_db = pd.concat([dataset_sc1, dataset_yas], ignore_index=True)

    print(f"\n Dataset DB chargé: {len(dataset_db)} exemples")
    print(f"   - SC1:  {len(dataset_sc1)}")
    print(f"   - YAS1: {len(dataset_yas)}")
    print(f" TestSet DB chargé: {len(testset_db)} exemples")
    print(f"\nColonnes dataset: {dataset_db.columns.tolist()}")
    print(f"Premières lignes:\n{dataset_db.head()}")
except FileNotFoundError as e:
    print(f"  Fichiers non trouvés: {e}")
    print(f"   Cherchez dans: {active_learning_dir}")

 Chemins:
   Dataset CSV (SC1): C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\Dataset\coords_SC1.csv
   Dataset CSV (YAS1): C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\Dataset\Coords_Yas1.csv
   TestSet CSV: C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\testSet1\coords_SC-test.csv

 Dataset DB chargé: 10716 exemples
   - SC1:  7149
   - YAS1: 3567
 TestSet DB chargé: 3569 exemples

Colonnes dataset: ['frame_id', 'timestamp', 'x', 'y', 'source', 'img_prefix']
Premières lignes:
   frame_id  timestamp     x    y source img_prefix
0         0   0.000000  1524  295    sc1        SC1
1         1   0.276893  1544  299    sc1        SC1
2         2   0.309939  1563  303    sc1        SC1
3         3   0.344043  1583  307    sc1        SC1
4         4   0.377526  1603  311    sc1        SC1


## Section 2: Charger le checkpoint du modèle (entraîné sur DA)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Sélection automatique de l'architecture
IMAGE_SIZE = 224

class GazeNetLarge(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 5, padding=2)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, 5, padding=2)
        self.fc1 = nn.Linear(64 * 56 * 56, 512)
        self.fc2 = nn.Linear(512, 2)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return torch.sigmoid(self.fc2(x))

class GazeNetSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 14 * 14, 256)
        self.fc2 = nn.Linear(256, 2)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return torch.sigmoid(self.fc2(x))

# Charger le checkpoint entraîné sur domaine A
checkpoint_path = os.path.join(active_learning_dir, 'data', 'gaze_model_trained.pth')
raw = torch.load(checkpoint_path, map_location=device)
state = raw['state_dict'] if isinstance(raw, dict) and 'state_dict' in raw else raw

# Détection automatique (Small vs Large)
conv1_w = state['conv1.weight']
fc1_w = state['fc1.weight']
use_small = (conv1_w.shape == torch.Size([16, 3, 3, 3]) and fc1_w.shape[1] == 6272)

if use_small:
    IMAGE_SIZE = 28
    ModelClass = GazeNetSmall
    model_name = 'GazeNetSmall (28x28 input)'
else:
    IMAGE_SIZE = 224
    ModelClass = GazeNetLarge
    model_name = 'GazeNetLarge (224x224 input)'

model = ModelClass().to(device)
model.load_state_dict(state)
model.eval()

print(f"Checkpoint: {checkpoint_path}")
print(f"Architecture: {model_name}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Modèle sur device: {device}")

 Checkpoint chargé depuis: C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\data\gaze_model_trained.pth
Modèle sur device: cpu


In [56]:
# VÉRIFICATION DU CHECKPOINT (demandée par le prof)
print("\n" + "="*60)
print(" VÉRIFICATION DÉTAILLÉE DU CHECKPOINT")
print("="*60)

# 1. Vérifier la taille du fichier
if os.path.exists(checkpoint_path):
    file_size = os.path.getsize(checkpoint_path) / (1024 * 1024)  # MB
    print(f"\n FICHIER:")
    print(f"   Chemin: {checkpoint_path}")
    print(f"   Taille: {file_size:.2f} MB")
    print(f"    Fichier existe")
else:
    print(f"    Fichier non trouvé!")

# 2. Compter les paramètres du modèle
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n ARCHITECTURE:")
print(f"   Paramètres totaux:       {total_params:,}")
print(f"   Paramètres entraînables: {trainable_params:,}")
print(f"   Taille mémoire:          ~{total_params * 4 / (1024*1024):.2f} MB")

# 3. Afficher les couches
print(f"\n COUCHES DU MODÈLE:")
for name, param in model.named_parameters():
    print(f"   {name:<15} - Shape: {str(list(param.shape)):<20} - {param.numel():>8,} params")

# 4. Test de prédiction (vérifier que ça marche)
print(f"\n TEST DE PRÉDICTION:")
test_input = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    test_output = model(test_input)

print(f"   Input shape:  {list(test_input.shape)}")
print(f"   Output shape: {list(test_output.shape)}")
print(f"   Output values: [{test_output[0, 0]:.4f}, {test_output[0, 1]:.4f}]")
print(f"   Output range:  [{test_output.min():.4f}, {test_output.max():.4f}]")

# 5. Validation finale
if test_output.shape == torch.Size([1, 2]):
    print(f"   Sortie correcte: 2 coordonnées (x, y)")
else:
    print(f"   Sortie incorrecte!")

if 0 <= test_output.min() and test_output.max() <= 1:
    print(f"   Valeurs dans [0,1] (sigmoid fonctionne)")
else:
    print(f"    Valeurs hors [0,1]!")

print(f"\n CHECKPOINT VALIDÉ ET PRÊT")
print(f"   Device: {device}")
print("="*60)



 VÉRIFICATION DÉTAILLÉE DU CHECKPOINT

 FICHIER:
   Chemin: C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\data\gaze_model_trained.pth
   Taille: 6.15 MB
    Fichier existe

 ARCHITECTURE:
   Paramètres totaux:       1,611,490
   Paramètres entraînables: 1,611,490
   Taille mémoire:          ~6.15 MB

 COUCHES DU MODÈLE:
   conv1.weight    - Shape: [16, 3, 3, 3]        -      432 params
   conv1.bias      - Shape: [16]                 -       16 params
   conv2.weight    - Shape: [32, 16, 3, 3]       -    4,608 params
   conv2.bias      - Shape: [32]                 -       32 params
   fc1.weight      - Shape: [256, 6272]          - 1,605,632 params
   fc1.bias        - Shape: [256]                -      256 params
   fc2.weight      - Shape: [2, 256]             -      512 params
   fc2.bias        - Shape: [2]                  -        2 params

 TEST DE PRÉDICTION:
   Input shape:  [1, 3, 224, 224]
   Output shape: [1, 2]
   Output values: [0.3221, 0.7722]
   O

## Section 3: Préparer les DataLoaders pour le domaine B

In [ ]:
# Dataset Domain B (supporte SC1 + YAS1 + TestSet)
class DomainBDataset(torch.utils.data.Dataset):
    def __init__(self, df, img_dir, default_prefix="SC1"):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.default_prefix = default_prefix
        # Normalisation auto si besoin
        self.max_x = float(self.df['x'].max())
        self.max_y = float(self.df['y'].max())
        self.min_x = float(self.df['x'].min())
        self.min_y = float(self.df['y'].min())
        self.is_normalized = (0.0 <= self.min_x <= 1.0 and 0.0 <= self.min_y <= 1.0 and
                              0.0 <= self.max_x <= 1.0 and 0.0 <= self.max_y <= 1.0)
        
    def __len__(self):
        return len(self.df)
    
    def _resolve_img_dir(self, row):
        if isinstance(self.img_dir, dict):
            source = str(row.get('source', 'sc1')).lower()
            return self.img_dir.get(source, next(iter(self.img_dir.values())))
        return self.img_dir
    
    def _resolve_prefix(self, row):
        if 'img_prefix' in row and pd.notna(row['img_prefix']):
            return str(row['img_prefix'])
        return self.default_prefix
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_dir = self._resolve_img_dir(row)
        prefix = self._resolve_prefix(row)
        img_path = os.path.join(img_dir, f"{prefix}_{int(row['frame_id']):04d}.png")
        
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        img = torch.from_numpy(img)
        
        x_val = float(row['x'])
        y_val = float(row['y'])
        if not self.is_normalized and self.max_x > 0 and self.max_y > 0:
            x_val = x_val / self.max_x
            y_val = y_val / self.max_y
        label = torch.tensor([x_val, y_val], dtype=torch.float32)
        return img, label

try:
    # Répertoires d'images
    dataset_imgs_dir = {
        'sc1': os.path.join(dataset_dir, 'dataStev', 'framesSC1'),
        'yas1': os.path.join(dataset_dir, 'frameYas1')
    }
    testset_imgs_dir = os.path.join(testset_dir, 'framesTest')
    
    dataset_db_loader = DomainBDataset(dataset_db, dataset_imgs_dir, default_prefix="SC1")
    testset_db_loader = DomainBDataset(testset_db, testset_imgs_dir, default_prefix="SC1")
    
    # DataLoaders
    batch_size = 32
    db_dataloader = DataLoader(dataset_db_loader, batch_size=batch_size, shuffle=False)
    testdb_dataloader = DataLoader(testset_db_loader, batch_size=batch_size, shuffle=False)
    
    print(f" DataLoaders créés")
    print(f"   Dataset DB: {len(dataset_db_loader)} exemples")
    print(f"   TestSet DB: {len(testset_db_loader)} exemples")
    print(f"   Images dir (Dataset SC1): {dataset_imgs_dir['sc1']}")
    print(f"   Images dir (Dataset YAS1): {dataset_imgs_dir['yas1']}")
    print(f"   Images dir (TestSet): {testset_imgs_dir}")
except Exception as e:
    print(f"  Erreur: {e}")
    import traceback
    traceback.print_exc()

 DataLoaders créés
   Dataset DB: 10716 exemples
   TestSet DB: 3569 exemples
   Images dir (Dataset SC1): C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\Dataset\dataStev\framesSC1
   Images dir (Dataset YAS1): C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\Dataset\frameYas1
   Images dir (TestSet): C:\Users\PC\OneDrive\Bureau\Projet_Active_learning\active_learning\testSet1\framesTest


## Section 4: Évaluer le checkpoint sur le domaine B (BASELINE)

In [59]:
def evaluate_model(model, dataloader, device):
    """Évalue le modèle sur un DataLoader"""
    model.eval()
    preds = []
    labels = []
    loss_fn = nn.MSELoss()
    total_loss = 0.0
    
    with torch.no_grad():
        for imgs, labs in dataloader:
            imgs = imgs.to(device)
            labs = labs.to(device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labs)
            total_loss += loss.item()
            preds.append(outputs.cpu())
            labels.append(labs.cpu())
    
    preds = torch.cat(preds)
    labels = torch.cat(labels)
    mae = torch.mean(torch.abs(preds - labels)).item()
    rmse = torch.sqrt(torch.mean((preds - labels)**2)).item()
    mse = total_loss / len(dataloader)
    
    return {
        'mse': mse,
        'mae': mae,
        'rmse': rmse,
        'preds': preds,
        'labels': labels
    }

# Évaluer sur testSet du domaine B
print("=" * 60)
print("BASELINE: Évaluation du checkpoint sur DOMAINE B")
print("=" * 60)
try:
    metrics = evaluate_model(model, testdb_dataloader, device)
    print(f"\n Résultats sur TestSet Domaine B:")
    print(f"   MSE:  {metrics['mse']:.6f}")
    print(f"   MAE:  {metrics['mae']:.6f} (≈ {metrics['mae']*((1272+712)/2):.2f} px)")
    print(f"   RMSE: {metrics['rmse']:.6f}")
    print(f"\n Cette baseline montre la performance SANS adaptation")
    print(f"    On va maintenant utiliser Active Learning pour l'améliorer.")
except Exception as e:
    print(f" Erreur lors de l'évaluation: {e}")

BASELINE: Évaluation du checkpoint sur DOMAINE B

 Résultats sur TestSet Domaine B:
   MSE:  0.064411
   MAE:  0.208175 (≈ 206.51 px)
   RMSE: 0.254045

 Cette baseline montre la performance SANS adaptation
    On va maintenant utiliser Active Learning pour l'améliorer.


# NOUVELLE EXPÉRIENCE: Active Learning Optimisé pour Régression

**Protocole:**
- Stratégies: Random, TTA Variance (uncertainty), K-means (diversity), Mixed (U+D), Ensemble
- Budgets: 1%, 2%, 10%, 20%, 50% (calculés sur le pool AL)
- Runs: 2 par stratégie
- Dataset split: 10% baseline train / 90% AL pool
- Training: 2 epochs, continuous, incremental selection

In [ ]:
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

class AL:
    def uncertainty_tta(self, m, l, d, n=3):
        """TTA variance = uncertainty"""
        m.eval()
        unc = []
        with torch.no_grad():
            for img, _ in l:
                img = img.to(d)
                p = []
                for _ in range(n):
                    if torch.rand(1) > 0.5:
                        img_aug = torch.flip(img, dims=[3])
                    else:
                        img_aug = img
                    p.append(m(img_aug).cpu().numpy())
                p = np.array(p)
                unc.extend(np.var(p, axis=0).mean(axis=1))
        return np.array(unc)
    
    def diversity_kmeans(self, m, l, d, n_c):
        """K-means for diversity"""
        m.eval()
        feat = []
        with torch.no_grad():
            for img, _ in l:
                img = img.to(d)
                x = m.pool(F.relu(m.conv1(img)))
                x = m.pool(F.relu(m.conv2(x)))
                x = x.view(x.size(0), -1)
                feat.append(x.cpu().numpy())
        feat = np.concatenate(feat)
        if feat.shape[1] > 50:
            pca = PCA(50)
            feat = pca.fit_transform(feat)
        km = KMeans(min(n_c, len(feat)), random_state=42)
        km.fit(feat)
        sel = []
        for i in range(km.n_clusters):
            idx = np.where(km.labels_==i)[0]
            if len(idx) > 0:
                d_to_c = np.linalg.norm(feat[idx] - km.cluster_centers_[i], axis=1)
                sel.append(idx[np.argmin(d_to_c)])
        return np.array(sel)
    
    def mixed_ud(self, m, l, d, a=0.6):
        """U + D mixed"""
        u = self.uncertainty_tta(m, l, d, n=3)
        u = (u - u.min()) / (u.max() - u.min() + 1e-8)
        m.eval()
        feat = []
        with torch.no_grad():
            for img, _ in l:
                img = img.to(d)
                x = m.pool(F.relu(m.conv1(img)))
                x = m.pool(F.relu(m.conv2(x)))
                x = x.view(x.size(0), -1)
                feat.append(x.cpu().numpy())
        feat = np.concatenate(feat)
        dist = cdist(feat, feat)
        div = np.mean(dist, axis=1)
        div = (div - div.min()) / (div.max() - div.min() + 1e-8)
        return a * u + (1-a) * div
    
    def ensemble(self, m, l, d, n=3):
        """Ensemble voting"""
        m.eval()
        ens = []
        with torch.no_grad():
            for img, _ in l:
                img = img.to(d)
                b = []
                for _ in range(n):
                    m.train()
                    b.append(m(img).cpu().numpy())
                ens.append(b)
        m.eval()
        all_p = []
        for b in ens:
            b = np.array(b)
            all_p.extend(np.var(b, axis=0).mean(axis=1))
        return np.array(all_p)

print("✓ AL strategies ready")

In [ ]:
# Configuration expérience optimisée
budgets_pct = [1, 2, 10, 20, 50]
n_runs = 2
num_epochs_al = 2
learning_rate_al = 1e-4
batch_size = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Protocole
incremental_selection = True
continuous_training = True

print(f"Configuration:")
print(f"  Budgets %: {budgets_pct}")
print(f"  Runs: {n_runs}")
print(f"  Epochs: {num_epochs_al}")
print(f"  Device: {device}")

Configuration:
  Budgets: [107, 1071, 5358] (['1.0%', '10.0%', '50.0%'])
  Runs: 2
  Epochs: 3
  Device: cpu


In [ ]:
# Split dataset: 10% baseline train / 90% AL pool
import time
from torch import nn, optim
import copy

split_ratio = 0.1
baseline_size = int(len(dataset_db) * split_ratio)
indices = np.random.RandomState(42).permutation(len(dataset_db))

baseline_train_db = dataset_db.iloc[indices[:baseline_size]].reset_index(drop=True)
al_pool_db = dataset_db.iloc[indices[baseline_size:]].reset_index(drop=True)

# Budgets basés sur la taille du pool
budgets = [max(1, int(len(al_pool_db) * p / 100)) for p in budgets_pct]
budgets = sorted(list(dict.fromkeys(budgets)))
budgets_pct_real = [round(b / len(al_pool_db) * 100, 1) for b in budgets]
print(f"Budgets (AL pool): {budgets} -> {budgets_pct_real}%")

print(f"Split dataset:")
print(f"  Baseline train: {len(baseline_train_db)} ({split_ratio*100:.0f}%)")
print(f"    - SC1:  {len(baseline_train_db[baseline_train_db['source']=='sc1'])}")
print(f"    - YAS1: {len(baseline_train_db[baseline_train_db['source']=='yas1'])}")
print(f"  AL pool: {len(al_pool_db)}")
print(f"  Test: {len(testset_db)}")

# Train baseline model on Domain B
print("\n Training baseline model on Domain B...")
start_time = time.time()

baseline_model_db = ModelClass().to(device)
baseline_model_db.load_state_dict(model.state_dict())

baseline_train_loader = DataLoader(
    DomainBDataset(baseline_train_db, dataset_imgs_dir),
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

criterion = nn.MSELoss()
optimizer = optim.Adam(baseline_model_db.parameters(), lr=learning_rate_al)

baseline_model_db.train()
for epoch in range(5):
    epoch_loss = 0.0
    for images, labels in baseline_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = baseline_model_db(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(baseline_train_loader)
    print(f"  Epoch {epoch+1}/5 - Loss: {avg_loss:.6f}")

# Evaluate baseline
baseline_metrics_db = evaluate_model(baseline_model_db, testdb_dataloader, device)
training_time = time.time() - start_time

print(f"\n✓ Baseline Domain B trained")
print(f"  Metrics: MSE={baseline_metrics_db['mse']:.4f}, MAE={baseline_metrics_db['mae']:.4f}, RMSE={baseline_metrics_db['rmse']:.4f}")
print(f"  Training time: {training_time:.1f}s")
print(f"  Model params: {sum(p.numel() for p in baseline_model_db.parameters()):,}")

# Save baseline state
baseline_state = baseline_model_db.state_dict()

# Training function
def train_model(model, train_df, num_epochs, device):
    model.train()
    train_loader = DataLoader(
        DomainBDataset(train_df, dataset_imgs_dir),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate_al)
    
    for epoch in range(num_epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model

Split dataset:
  Baseline train: 1071 (10%)
    - SC1:  733
    - YAS1: 338
  AL pool: 9645
  Test: 3569

 Training baseline model on Domain B...
  Epoch 1/5 - Loss: 0.056854
  Epoch 2/5 - Loss: 0.055961
  Epoch 3/5 - Loss: 0.055733
  Epoch 4/5 - Loss: 0.055443
  Epoch 5/5 - Loss: 0.055050

✓ Baseline Domain B trained
  Metrics: MSE=0.0648, MAE=0.2087, RMSE=0.2548
  Training time: 211.1s
  Model params: 1,611,490


In [ ]:
# EXPÉRIENCE OPTIMISÉE (5 stratégies, budgets %, 2 runs)
al = AL()

pool_loader = DataLoader(
    DomainBDataset(al_pool_db, dataset_imgs_dir),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

strategies = ['random', 'uncertainty_tta', 'diversity_kmeans', 'mixed_ud', 'ensemble']
results = {s: {b: [] for b in budgets} for s in strategies}
selection_cache = {}

for run in range(n_runs):
    print("\n" + "="*80)
    print(f"RUN {run+1}/{n_runs}")
    print("="*80)

    np.random.seed(100 + run)
    torch.manual_seed(100 + run)

    scoring_model = ModelClass().to(device)
    scoring_model.load_state_dict(baseline_state)

    # Pré-calculer rankings
    scores = {}
    scores['random'] = np.random.permutation(len(al_pool_db))

    print("  Scoring TTA...")
    scores['uncertainty_tta'] = np.argsort(-al.uncertainty_tta(scoring_model, pool_loader, device, n=3))

    print("  Scoring K-means...")
    div_idx = al.diversity_kmeans(scoring_model, pool_loader, device, n_c=len(budgets)*2)
    scores['diversity_kmeans'] = div_idx

    print("  Scoring Mixed U+D...")
    scores['mixed_ud'] = np.argsort(-al.mixed_ud(scoring_model, pool_loader, device, a=0.6))

    print("  Scoring Ensemble...")
    scores['ensemble'] = np.argsort(-al.ensemble(scoring_model, pool_loader, device, n=3))

    for strategy in strategies:
        print(f"\n--- STRATEGY: {strategy} ---")
        current_state = baseline_state if continuous_training else None
        selected_so_far = []
        ranking = scores[strategy]
        if strategy == 'diversity_kmeans' and len(ranking) < len(al_pool_db):
            ranking = list(ranking) + [i for i in range(len(al_pool_db)) if i not in set(ranking)]

        for b_idx, budget in enumerate(budgets):
            if incremental_selection:
                need = budget - len(selected_so_far)
                if need > 0:
                    add = [idx for idx in ranking if idx not in selected_so_far][:need]
                    selected_so_far.extend(add)
                selected_idx = selected_so_far
            else:
                selected_idx = list(ranking[:budget])

            selected_df = al_pool_db.iloc[selected_idx].copy()
            train_df = pd.concat([baseline_train_db, selected_df], ignore_index=True)

            model_train = ModelClass().to(device)
            if continuous_training and incremental_selection and current_state is not None:
                model_train.load_state_dict(current_state)
            else:
                model_train.load_state_dict(baseline_state)

            train_loader = DataLoader(
                DomainBDataset(train_df, dataset_imgs_dir),
                batch_size=batch_size,
                shuffle=True,
                num_workers=0
)
            
            criterion_train = nn.MSELoss()
            optimizer_train = optim.Adam(model_train.parameters(), lr=learning_rate_al)
            model_train.train()

            for epoch in range(num_epochs_al):
                for images, labels in train_loader:
                    images, labels = images.to(device), labels.to(device)
                    optimizer_train.zero_grad()
                    outputs = model_train(images)
                    loss = criterion_train(outputs, labels)
                    loss.backward()
                    optimizer_train.step()

            metrics = evaluate_model(model_train, testdb_dataloader, device)
            results[strategy][budget].append(metrics['mae'])

            pct = budgets_pct_real[b_idx] if 'budgets_pct_real' in globals() else (budget/len(al_pool_db)*100)
            print(f"Budget {pct:.1f}% ({budget:5d}) -> MAE: {metrics['mae']:.6f}")

            if continuous_training and incremental_selection:
                current_state = model_train.state_dict()

        if run == 0:
            selection_cache[strategy] = selected_df

# Résumé
print("\n" + "="*80)
print(f"RÉSULTATS MOYENS (MAE) SUR {n_runs} RUNS")
print("="*80)

summary = []
for strategy in strategies:
    for budget in budgets:
        maes = results[strategy][budget]
        mean_mae = np.mean(maes)
        std_mae = np.std(maes)
        summary.append({
            'strategy': strategy,
            'budget': budget,
            'mean_mae': mean_mae,
            'std_mae': std_mae
        })
        print(f"{strategy:<16} | Budget {budget:5d} -> {mean_mae:.6f} ± {std_mae:.6f}")

summary_df = pd.DataFrame(summary)


RUN 1/2
  Scoring TTA variance...
  Scoring mixed strategy...

--- STRATEGY: random ---
Budget   107 -> MAE: 0.209465
Budget  1071 -> MAE: 0.213706
Budget  5358 -> MAE: 0.218068

--- STRATEGY: tta_var ---
Budget   107 -> MAE: 0.209774
Budget  1071 -> MAE: 0.214325
Budget  5358 -> MAE: 0.219220

--- STRATEGY: mixed_ud ---
Budget   107 -> MAE: 0.209483
Budget  1071 -> MAE: 0.214854
Budget  5358 -> MAE: 0.217238

RUN 2/2
  Scoring TTA variance...
  Scoring mixed strategy...

--- STRATEGY: random ---
Budget   107 -> MAE: 0.209460
Budget  1071 -> MAE: 0.212163
Budget  5358 -> MAE: 0.219987

--- STRATEGY: tta_var ---
Budget   107 -> MAE: 0.209400
Budget  1071 -> MAE: 0.214943



RUN 1/2
  Scoring TTA variance...
  Scoring mixed strategy...

--- STRATEGY: random ---
Budget   107 -> MAE: 0.209465
Budget  1071 -> MAE: 0.213706
Budget  5358 -> MAE: 0.218068

--- STRATEGY: tta_var ---
Budget   107 -> MAE: 0.209774
Budget  1071 -> MAE: 0.214325
Budget  5358 -> MAE: 0.219220

--- STRATEGY: mixed_ud ---
Budget   107 -> MAE: 0.209483
Budget  1071 -> MAE: 0.214854
Budget  5358 -> MAE: 0.217238

RUN 2/2
  Scoring TTA variance...
  Scoring mixed strategy...

--- STRATEGY: random ---
Budget   107 -> MAE: 0.209460
Budget  1071 -> MAE: 0.212163
Budget  5358 -> MAE: 0.219987

--- STRATEGY: tta_var ---
Budget   107 -> MAE: 0.209400
Budget  1071 -> MAE: 0.214943


KeyboardInterrupt: 

In [ ]:
# GRAPHES PERFORMANCE vs COST
fig, ax = plt.subplots(figsize=(14, 8))

colors = {
    'random': '#7f7f7f',
    'uncertainty_tta': '#1f77b4',
    'diversity_kmeans': '#ff7f0e',
    'mixed_ud': '#2ca02c',
    'ensemble': '#d62728'
}
markers = {
    'random': 's',
    'uncertainty_tta': 'o',
    'diversity_kmeans': '^',
    'mixed_ud': 'D',
    'ensemble': 'v'
}
labels = {
    'random': 'Random',
    'uncertainty_tta': 'TTA (Uncertainty)',
    'diversity_kmeans': 'K-means (Diversity)',
    'mixed_ud': 'Mixed (U+D)',
    'ensemble': 'Ensemble'
}

ax.axhline(y=baseline_metrics_db['mae'], color='red', linestyle=':', 
           linewidth=2, alpha=0.7, label='Baseline (10% data)')

for strategy in strategies:
    means = [np.mean(results[strategy][b]) for b in budgets]
    stds = [np.std(results[strategy][b]) for b in budgets]
    ci95 = [1.96 * s / np.sqrt(n_runs) for s in stds]
    
    ax.plot(budgets, means, marker=markers[strategy], 
            color=colors[strategy], linewidth=2.5, markersize=10,
            label=labels[strategy], markeredgewidth=1.5, markeredgecolor='black')
    
    ax.fill_between(budgets,
                     [means[i] - ci95[i] for i in range(len(budgets))],
                     [means[i] + ci95[i] for i in range(len(budgets))],
                     alpha=0.15, color=colors[strategy])

ax.set_xlabel('AL-selected samples', fontsize=13, fontweight='bold')
ax.set_ylabel('MAE (Lower is Better)', fontsize=13, fontweight='bold')
ax.set_title('Active Learning: Performance vs Budget (Mean ± 95% CI)', fontsize=15, fontweight='bold')
ax.set_xticks(budgets)
if 'budgets_pct_real' in globals():
    ax.set_xticklabels([f"{p}%\n({b})" for p, b in zip(budgets_pct_real, budgets)])
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='best', framealpha=0.95)

plt.tight_layout()
plt.savefig(os.path.join(active_learning_dir, 'al_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# Table résumé
print("\n" + "="*60)
print(f"{'Strategy':<20} {'Budget':<12} {'MAE Mean':<12} {'Std':<10}")
print("="*60)
for strategy in strategies:
    for budget in budgets:
        mean_mae = np.mean(results[strategy][budget])
        std_mae = np.std(results[strategy][budget])
        print(f"{strategy:<20} {budget:<12} {mean_mae:.6f}     {std_mae:.6f}")
print("="*60)